In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
!curl -SOL https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1089k 100  1089k   0     0  6981k     0  --:--:-- --:--:-- --:--:--  7027k


In [3]:
def tokenize(s):
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    tokens = re.findall(r"[a-z]+|[0-9]+|[^\w\s]", s)
    return tokens

In [4]:
with open("input.txt") as f:
    dataset = f.read()

dataset = tokenize(dataset)
print(f"Input tokens: {len(dataset)}")
vocab = list(set(dataset)) + ["<PAD>", "<UNK>"]
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

token_to_id = {t: i for i, t in enumerate(vocab)}
id_to_token = {i: t for t, i in token_to_id.items()}

Input tokens: 262927
Vocab size: 11468


In [5]:
ids = [token_to_id.get(w, token_to_id["<UNK>"]) for w in dataset]
split = int(0.9 * len(ids))
train_ids = torch.tensor(ids[:split], dtype=torch.long)
test_ids  = torch.tensor(ids[split:], dtype=torch.long)
def get_batch(data_ids, batch_size, seq_len):
    max_start = len(data_ids) - (seq_len + 1)
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([
        data_ids[s:s + seq_len]
        for s in starts
    ])
    y = torch.stack([
        data_ids[s + 1:s + 1 + seq_len]
        for s in starts
    ])
    return x.to(device), y.to(device)

def get_bilm_batch(data_ids, batch_size, seq_len):
    L = len(data_ids)
    max_start = L - (seq_len + 1)
    assert max_start > 0, "Data too small for the chosen seq_len."

    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data_ids[int(s.item()): int(s.item()) + seq_len] for s in starts])
    y_next = torch.stack([data_ids[int(s.item()) + 1: int(s.item()) + 1 + seq_len] for s in starts])

    x_rev = torch.flip(x, dims=[1])
    y_next_rev = torch.flip(y_next, dims=[1])

    return x.to(device), y_next.to(device), x_rev.to(device), y_next_rev.to(device)

In [6]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)
    def forward(self, x):
        embed = self.embedding(x)
        final_h, _ = self.lstm(embed)
        return self.output(final_h), final_h
    def do_train(self, num_iters, alpha):
        opt = torch.optim.Adam(self.parameters(), lr=alpha)
        self.train()
        print("Training", end="")
        for i in range(num_iters):
            x, y = get_batch(train_ids, 128, 32)
            logits, _ = self(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            opt.step()
            if i % (num_iters // 100) == 0:
                print(".", end="")
        print("Done")

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm_f = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.lstm_r = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.output_f = nn.Linear(hidden_size, vocab_size)
        self.output_r = nn.Linear(hidden_size, vocab_size)
    def forward(self, x):
        embed = self.embedding(x)
        final_h_f, _ = self.lstm_f(embed)
        out_f = self.output_f(final_h_f)

        embed_r = torch.flip(embed, dims=[1])
        final_h_r, _ = self.lstm_r(embed_r)
        out_r = self.output_r(final_h_r)

        return out_f, out_r, torch.cat([final_h_f, final_h_r], dim=-1)
    def do_train(self, num_iters, alpha):
        opt = torch.optim.Adam(self.parameters(), lr=alpha)
        self.train()
        print("Training", end="")
        for i in range(num_iters):
            x, y_next, x_rev, y_next_rev = get_bilm_batch(train_ids, 128, 32)
            logits_fwd, logits_bwd, _ = self(x)
        
            loss_f = F.cross_entropy(logits_fwd.reshape(-1, vocab_size), y_next.reshape(-1))
            loss_b = F.cross_entropy(logits_bwd.reshape(-1, vocab_size), y_next_rev.reshape(-1))
            loss = loss_f + loss_b
        
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            opt.step()
            if i % (num_iters // 100) == 0:
                print(".", end="")
        print("Done")

In [7]:
model1 = LSTMLM(vocab_size).to(device)
model1.do_train(5000, 1e-3)

Training....................................................................................................Done


In [8]:
model2 = BiLSTM(vocab_size).to(device)
model2.do_train(5000, 1e-3)

Training....................................................................................................Done
